# Feature Engineering & Model Preparation

This notebook consumes the cleaned baseline dataset produced by `Data_Cleaning_and_Processing_Clean.ipynb` and adds derived features plus modeling prep steps.

**Pipeline (this notebook):**
1. Load cleaned baseline dataset + manifest
2. Basic integrity checks (schema, dtypes)
3. ZIP normalization (idempotent)
4. Derived metrics (PRICE_PER_SQFT, LOG_PRICE, LOG_PRICE_PER_SQFT)
5. Categorical frequency capping (SUBLOCALITY, PROPERTY_TYPE, TYPE) with *_raw backups
6. Outlier diagnostics (IQR*3 counts) without removal (optional removal toggle)
7. Train/Validation split seed reproducible
8. Export engineered dataset + updated manifest
9. (Optional) Simple baseline model (LinearRegression / RandomForest) scaffold

---

In [ ]:
# 1. Imports & Paths
import pandas as pd, numpy as np, json, os, math
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
DATA_DIR = Path('Resources')
CLEAN_BASELINE_FILE = DATA_DIR / 'NY-House-Dataset-Cleaned.csv'
ENGINEERED_FILE = DATA_DIR / 'NY-House-Dataset-Engineered.csv'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('Paths set.')

Paths set.


In [22]:
# 2. Load Clean Baseline + Manifest
df = pd.read_csv(CLEAN_BASELINE_FILE)
with open(BASELINE_MANIFEST, 'r', encoding='utf-8') as f: baseline_manifest = json.load(f)
print('Loaded baseline rows:', len(df))
print('Baseline columns:', df.columns.tolist())
display(df.head(3))

Loaded baseline rows: 4039
Baseline columns: ['TYPE', 'PRICE', 'BEDROOMS', 'BATHROOMS', 'SQFT', 'LATITUDE', 'LONGITUDE', 'BOROUGH_orig', 'SUBLOCALITY_ext', 'ZIPCODE', 'PROPERTY_TYPE', 'BOROUGH', 'PRICE_PER_SQFT', 'LOG_PRICE']


,TYPE,PRICE,BEDROOMS,BATHROOMS,SQFT,LATITUDE,LONGITUDE,BOROUGH_orig,SUBLOCALITY_ext,ZIPCODE,PROPERTY_TYPE,BOROUGH,PRICE_PER_SQFT,LOG_PRICE
0,condo,315000,2,2.0,1400,40.761255,-73.974483,Manhattan,midtown east,10022,amenity,manhattan,225.000000,12.660331
1,house,260000,4,2.0,2015,40.541805,-74.196109,Richmond County,richmond county,10312,building,staten island,129.032258,12.468441
2,condo,69000,3,1.0,445,40.761398,-73.974613,New York County,midtown east,10022,amenity,manhattan,155.056180,11.141876


In [23]:
# 3. Integrity Checks
expected_min_cols = {'PRICE','SQFT','LATITUDE','LONGITUDE'}
missing_expected = expected_min_cols.difference(df.columns)
assert not missing_expected, f'Missing expected baseline columns: {missing_expected}'
assert df['PRICE'].gt(0).all(), 'PRICE must be positive.'
if 'SQFT' in df: assert df['SQFT'].gt(0).all(), 'SQFT must be positive.'
print('Integrity checks passed.')

Integrity checks passed.


In [24]:
# 4. ZIP Normalization (Idempotent)
if 'ZIPCODE' in df:
    df['ZIPCODE'] = df['ZIPCODE'].astype(str).str.extract(r'(\d{5})')[0].fillna('').str.zfill(5)
print('ZIP normalization complete.')

ZIP normalization complete.


In [25]:
# 5. Derived Features
if {'PRICE','SQFT'}.issubset(df.columns):
    df['PRICE_PER_SQFT'] = (df['PRICE']/df['SQFT']).replace([np.inf,-np.inf], np.nan)
    df.loc[(df['PRICE_PER_SQFT'] <= 0) | (df['PRICE_PER_SQFT']>50000), 'PRICE_PER_SQFT'] = np.nan
if 'PRICE_PER_SQFT' in df:
    df['LOG_PRICE_PER_SQFT'] = np.log1p(df['PRICE_PER_SQFT'])
if 'PRICE' in df:
    df['LOG_PRICE'] = np.log1p(df['PRICE'])
print('Derived numerical features added.')
display(df[['PRICE','SQFT','PRICE_PER_SQFT','LOG_PRICE','LOG_PRICE_PER_SQFT']].head())

Derived numerical features added.


,PRICE,SQFT,PRICE_PER_SQFT,LOG_PRICE,LOG_PRICE_PER_SQFT
0,315000,1400,225.000000,12.660331,5.420535
1,260000,2015,129.032258,12.468441,4.867783
2,69000,445,155.056180,11.141876,5.050216
3,690000,4004,172.327672,13.444448,5.155184
4,899500,2184,411.858974,13.709595,6.023106


In [26]:
# 6. Categorical Frequency Capping (with raw backups)
for c in ['SUBLOCALITY','PROPERTY_TYPE','TYPE']:
    if c in df.columns:
        raw_col = c + '_raw'
        if raw_col not in df:
            df[raw_col] = df[c]
        top = df[c].value_counts().nlargest(50).index
        df[c] = df[c].where(df[c].isin(top), 'other')
print('Categorical capping done.')
display(df.head(3))

Categorical capping done.


,TYPE,PRICE,BEDROOMS,BATHROOMS,SQFT,LATITUDE,LONGITUDE,BOROUGH_orig,SUBLOCALITY_ext,ZIPCODE,PROPERTY_TYPE,BOROUGH,PRICE_PER_SQFT,LOG_PRICE,LOG_PRICE_PER_SQFT,PROPERTY_TYPE_raw,TYPE_raw
0,condo,315000,2,2.0,1400,40.761255,-73.974483,Manhattan,midtown east,10022,amenity,manhattan,225.000000,12.660331,5.420535,amenity,condo
1,house,260000,4,2.0,2015,40.541805,-74.196109,Richmond County,richmond county,10312,building,staten island,129.032258,12.468441,4.867783,building,house
2,condo,69000,3,1.0,445,40.761398,-73.974613,New York County,midtown east,10022,amenity,manhattan,155.056180,11.141876,5.050216,amenity,condo


In [27]:
# 7. Outlier Diagnostics (IQR*3 counts, no removal by default)
def outlier_counts(series, factor=3.0):
    s = series.dropna()
    if s.empty: return 0
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - factor*iqr, q3 + factor*iqr
    return int(((s < lower) | (s > upper)).sum())
for c in ['PRICE','SQFT','PRICE_PER_SQFT']:
    if c in df:
        print(f'{c} outliers (IQR*3 rule):', outlier_counts(df[c]))
print('Outlier diagnostics complete.')

PRICE outliers (IQR*3 rule): 196
SQFT outliers (IQR*3 rule): 90
PRICE_PER_SQFT outliers (IQR*3 rule): 100
Outlier diagnostics complete.


In [28]:
# 8. Train / Validation Split (simple example for modeling scaffold)
target = 'LOG_PRICE' if 'LOG_PRICE' in df else 'PRICE'
feature_candidates = [c for c in df.columns if c not in {target,'LATITUDE','LONGITUDE'} and not c.endswith('_raw')]
# Very lightweight numeric-only example; advanced encoding handled later
X = df[[c for c in feature_candidates if df[c].dtype != 'object']].copy()
y = df[target].copy()
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)
print('Train/Val shapes:', X_train.shape, X_val.shape)
if X_train.shape[1] > 0:
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    preds = lr.predict(X_val)
    print('Baseline LinearRegression MAE:', mean_absolute_error(y_val, preds))
    print('Baseline LinearRegression R2:', r2_score(y_val, preds))
else:
    print('No numeric predictors available for quick baseline model.')

Train/Val shapes: (3231, 6) (808, 6)
Baseline LinearRegression MAE: 0.16610629517410777
Baseline LinearRegression R2: 0.9199656426771449


In [29]:
# 9. Export Engineered Dataset + Manifest
engineered_manifest = {
    'stage': 'feature_engineering',
    'rows': int(len(df)),
    'cols': int(df.shape[1]),
    'columns': df.columns.tolist(),
    'exported_at': pd.Timestamp.utcnow().isoformat(),
    'sample_head': df.head(5).to_dict(orient='records')
}
df.to_csv(ENGINEERED_FILE, index=False)
with open(ENGINEERED_MANIFEST,'w',encoding='utf-8') as f: json.dump(engineered_manifest, f, indent=2)
print('Engineered dataset exported ->', ENGINEERED_FILE)
print('Manifest written ->', ENGINEERED_MANIFEST)
df.head()

Engineered dataset exported -> Resources\NY-House-Dataset-Engineered.csv
Manifest written -> Resources\NY-House-Dataset-Engineered-manifest.json


,TYPE,PRICE,BEDROOMS,BATHROOMS,SQFT,LATITUDE,LONGITUDE,BOROUGH_orig,SUBLOCALITY_ext,ZIPCODE,PROPERTY_TYPE,BOROUGH,PRICE_PER_SQFT,LOG_PRICE,LOG_PRICE_PER_SQFT,PROPERTY_TYPE_raw,TYPE_raw
0,condo,315000,2,2.0,1400,40.761255,-73.974483,Manhattan,midtown east,10022,amenity,manhattan,225.000000,12.660331,5.420535,amenity,condo
1,house,260000,4,2.0,2015,40.541805,-74.196109,Richmond County,richmond county,10312,building,staten island,129.032258,12.468441,4.867783,building,house
2,condo,69000,3,1.0,445,40.761398,-73.974613,New York County,midtown east,10022,amenity,manhattan,155.056180,11.141876,5.050216,amenity,condo
3,house,690000,5,2.0,4004,40.674363,-73.958725,Kings County,kings county,11238,building,brooklyn,172.327672,13.444448,5.155184,building,house
4,condo,899500,2,2.0,2184,40.809448,-73.946777,New York,harlem,10027,building,manhattan,411.858974,13.709595,6.023106,building,condo


## Next Steps
- Advanced encoding (categorical embeddings / one-hot / target encoding).
- Feature selection (mutual information, SHAP, permutation importance).
- Model comparison (tree-based ensembles, regularized linear, gradient boosting).
- Hyperparameter tuning & cross-validation.

End of feature engineering & model prep scaffold.